In [9]:
#%pip install pandas
%pip freeze > requirements.txt

Note: you may need to restart the kernel to use updated packages.


### Verificación que el notebook se ejecuta dentro del ipykernel que se encuentra dentro del .venv, Si la ruta apunta a tu .venv, estás instalando dentro de tu entorno virtual de este proyecto, en este punto se verifica que tanto el interprete de python como el ipykernel se encuentren dentro de .venv, tener en cuenta que el ipykernel unicamente se puede instalar por consola despues de haber habilitado el entorno virtual

In [2]:
import sys, ipykernel
print("PYTHON =", sys.executable)
print("IPYKERNEL =", ipykernel.__file__)

PYTHON = d:\OneDrive - Grupo EPM\8. UNIVERSIDAD JAVERIANA\Applied Project I\Project\Data Collection\.venv\Scripts\python.exe
IPYKERNEL = d:\OneDrive - Grupo EPM\8. UNIVERSIDAD JAVERIANA\Applied Project I\Project\Data Collection\.venv\Lib\site-packages\ipykernel\__init__.py


In [1]:
# Leer archivo CSV desde carpeta local sincronizada de OneDrive (configurada en .env)
import os
import pandas as pd
from pathlib import Path
import logging
import sys

# Configuracion del logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[logging.StreamHandler(sys.stdout)]
)

def load_env_file(env_path='.env'):
    """Carga variables KEY=VALUE desde .env sin dependencias externas."""
    env_file = Path(env_path)
    if not env_file.exists():
        logging.warning(f'No se encontro el archivo de entorno: {env_file.resolve()}')
        return

    for line in env_file.read_text(encoding='utf-8').splitlines():
        raw = line.strip()
        if not raw or raw.startswith('#') or '=' not in raw:
            continue

        key, value = raw.split('=', 1)
        key = key.strip()
        value = value.strip().strip('"').strip("'")

        if key and key not in os.environ:
            os.environ[key] = value

def read_csv_from_env_location(csv_filename, env_var='UBICACION_DATA', encoding='utf-8-sig'):
    base_path = os.getenv(env_var)
    if not base_path:
        logging.error(f'La variable {env_var} no esta definida en el entorno/.env.')
        return None

    file_path = Path(base_path) / csv_filename
    if not file_path.exists():
        logging.error(f'No se encontro el archivo CSV en: {file_path}')
        return None

    try:
        df = pd.read_csv(file_path, encoding=encoding)
        logging.info(f'Archivo CSV leido exitosamente desde ruta local: {file_path}')
        return df
    except Exception as e:
        logging.error(f'No fue posible leer el CSV local: {e}')
        return None

# Ejemplo de uso
if __name__ == '__main__':
    load_env_file('.env')

    # El nombre del archivo es comun para todo el equipo; cambia solo UBICACION_DATA en .env
    csv_filename = 'tabla_eventos.csv'

    df = read_csv_from_env_location(csv_filename)
    if df is not None:
        df.head()
df.head()

2026-04-14 17:18:18,890 - WARNING - No se encontro el archivo de entorno: D:\OneDrive - Grupo EPM\8. UNIVERSIDAD JAVERIANA\Applied Project I\Project\Data Collection\Scripts\.env
2026-04-14 17:18:19,013 - INFO - Archivo CSV leido exitosamente desde ruta local: D:\OneDrive - PUJ Cali\Projecto_Grado_Javeriana\Data_Project\tabla_eventos.csv


,ID_EVENTO,FECHA_DESCONEXION,FECHA_CONEXION,ELEMENTO_FALLADO
0,801851,14/03/2021 06:59,14/03/2021 09:30,PSW9292
1,787583,04/01/2021 08:50,04/01/2021 17:15,CMVE2356
2,787496,03/01/2021 14:18,03/01/2021 14:40,RC0158
3,787232,01/01/2021 10:29,01/01/2021 13:30,INSC92
4,787455,03/01/2021 10:34,03/01/2021 13:30,RC0259


# Eliminacion de caracter P a algunos registros de "ELEMENTO_FALLADO"

In [2]:
# elimnar el caracter "P" cuando se encuentra como ultimo caracter de la columna "ELEMENTO_FALLADO" en el dataframe df
df['ELEMENTO_FALLADO'] = df['ELEMENTO_FALLADO'].str.rstrip('P') 
df.head()

,ID_EVENTO,FECHA_DESCONEXION,FECHA_CONEXION,ELEMENTO_FALLADO
0,801851,14/03/2021 06:59,14/03/2021 09:30,PSW9292
1,787583,04/01/2021 08:50,04/01/2021 17:15,CMVE2356
2,787496,03/01/2021 14:18,03/01/2021 14:40,RC0158
3,787232,01/01/2021 10:29,01/01/2021 13:30,INSC92
4,787455,03/01/2021 10:34,03/01/2021 13:30,RC0259


# Guardarlo en .csv en la carpeta 

In [3]:
# guardar en disco con pandas el dataframe df en formato csv con encoding utf-8-sig y sin indice, en la misma ruta local de origen pero con el nombre "tabla_eventos_ajustado.csv"
df.to_csv(Path(os.getenv('UBICACION_DATA')) / 'tabla_eventos_ajustado.csv', encoding='utf-8-sig', index=False)